# 35 孟加拉国 DIA：SIRIUS 证据分层与 PBT 候选最终表

本 Notebook 只读取已经冻结的 Notebook 34 结果，不再调用 SIRIUS，也不改变 Notebook 31 的模型排序。主要任务：

1. 对 Notebook 34 冻结的 458 个 feature（437 个 SIRIUS 支持谱图及 21 个多电荷审计谱图）做保守证据分层；
2. 显式标记结构候选分子式与 SIRIUS 排名第一分子式的不一致；
3. 保留全部跨成员注释冲突，不强行选择身份；冲突数量由 Notebook 34 冻结结果动态确定；
4. 生成 400 个冻结 cluster 的最终证据表、TOP_PRIORITY 验证表和冲突复核队列。

> **声明边界**：这里的 higher/moderate/low support 是内部操作性分层，不是校准概率。CSI:FingerID 候选最高仍为 Level 3 暂定结构；模型分数是探索性优先级，不是法规 PBT 判定。


In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'pbt_msms').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'pbt_msms').is_dir(), '没有找到项目根目录 src/pbt_msms'
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('ROOT =', ROOT)
print('Kernel: 请确认使用 Python (env_ms)')


In [ ]:
from pbt_msms.bangladesh_dia_evidence import paths, sha256_file
p = paths(ROOT)
manifest34 = json.loads(p['run_manifest34'].read_text(encoding='utf-8'))
assert manifest34['status'] == 'SIRIUS_ANNOTATIONS_IMPORTED_AND_FROZEN'
assert manifest34['n_features'] > 0 and manifest34['n_clusters'] == 400
for name, details in manifest34['artifacts'].items():
    artifact = p['out34'] / name
    assert artifact.is_file(), f'缺失 Notebook 34 文件: {name}'
    assert sha256_file(artifact) == details['sha256'], f'Notebook 34 文件哈希改变: {name}'
print('Notebook 34 frozen gate: PASS')
print('features =', manifest34['n_features'], '| clusters =', manifest34['n_clusters'])
print('formula success =', manifest34['formula_success'], '| structure success =', manifest34['structure_success'])


## 操作性证据分层

结构候选只有同时满足以下条件才进入较高或中等支持层：

- rank-1 结构的分子式与 SIRIUS 排名第一分子式一致；
- 指纹 Tanimoto、与最佳不同连接结构的 CSI 分差、SIRIUS normalized score 达到固定阈值；
- cluster 内不同成员没有结构或分子式冲突。

阈值仅用于安排后续标准品验证顺序，不可解释为鉴定置信概率。


In [ ]:
import importlib
import pbt_msms.bangladesh_dia_evidence as bd35
bd35 = importlib.reload(bd35)
manifest = bd35.run_evidence(ROOT)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
assert manifest['status'] == 'FINAL_EVIDENCE_TABLE_FROZEN'
assert manifest['n_features'] == manifest34['n_features']
assert manifest['n_clusters'] == manifest34['n_clusters']
assert manifest['n_top_priority_clusters'] == 100
assert 0 <= manifest['n_conflict_clusters'] <= manifest['n_clusters']


In [ ]:
import pandas as pd
OUT = p['out']
features = pd.read_csv(OUT / '35_feature_annotation_evidence.csv.gz')
clusters = pd.read_csv(OUT / '35_cluster_final_evidence_table.csv')
audit = pd.read_csv(OUT / '35_cohort_final_audit.csv')
top100 = pd.read_csv(OUT / '35_top_priority_verification_table.csv')
conflicts = pd.read_csv(OUT / '35_conflict_review_queue.csv')
assert len(features) == manifest['n_features'] and features['spectrum_id'].is_unique
assert len(clusters) == manifest['n_clusters'] and clusters['cross_polarity_cluster'].is_unique
assert len(top100) == manifest['n_top_priority_clusters'] and len(conflicts) == manifest['n_conflict_clusters']
assert audit['n_clusters'].eq(100).all()
assert conflicts['reported_inchikey_2d'].fillna('').eq('').all(), '冲突 cluster 不得强行报告结构身份'
strong = clusters['operational_evidence_tier'].isin(['STRUCTURE_HIGHER_SUPPORT','STRUCTURE_MODERATE_SUPPORT'])
assert clusters.loc[strong, 'cross_member_annotation_conflict'].eq(False).all()
assert clusters.loc[strong, 'representative_structure_formula_matches_top_sirius_formula'].eq(True).all()
display(audit)
display(clusters['operational_evidence_tier'].value_counts().rename_axis('tier').reset_index(name='n_clusters'))
print('Notebook 35 accounting and claim-boundary audit: PASS')


In [ ]:
show_columns = [
    'cluster_rank','cross_polarity_cluster','priority_score_primary','hazard_component',
    'operational_evidence_tier','identity_resolution_status','reported_structure_name',
    'reported_formula','representative_tanimoto_similarity',
    'representative_distinct_csi_margin','recommended_follow_up'
]
display(top100[show_columns].head(30))


In [ ]:
conflict_columns = [
    'cluster_rank','cross_polarity_cluster','cohort','member_spectrum_ids',
    'structure_candidates_across_members','formula_candidates_across_members',
    'representative_spectrum_id','representative_identity_caution_flags','recommended_follow_up'
]
display(conflicts[conflict_columns])


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUT / '35_cohort_evidence_distribution.png')))
display(Image(filename=str(OUT / '35_priority_annotation_map.png')))


## 完成标准

最后一个审计单元必须显示 `Notebook 35 accounting and claim-boundary audit: PASS`。

主要产物位于：`external_data/bangladesh_surface_water/06_revealed_evaluation/35_final_evidence_v1/`。

论文表述必须保持：Level 3 暂定结构、Level 4 分子式、Level 5 精确质量；需要标准品 RT 与 MS/MS 才能升级为 Level 1。
